# Ingredient / Component Instance Segmentation (YOLOv8-seg) — Kaggle Training

This notebook trains a YOLOv8 **instance segmentation** model (YOLOv8-seg) on a dataset that already has **segmentation polygons** in YOLO format, and saves the trained weights into **Kaggle Output** so you can download them.

## Dataset (works immediately on Kaggle)
Recommended dataset (already YOLO formatted):
- https://www.kaggle.com/datasets/tgiahuy/foodseg103-yolo2

> FoodSeg103 is “food component categories” (103 classes). It’s not a perfect ingredient vocabulary for every dish, but it’s a clean end-to-end demo for training instance segmentation on Kaggle.

## Kaggle steps
1. Kaggle → **New Notebook**
2. **File → Import Notebook → Upload** → upload this notebook (`07_kaggle_train_ingredient_yolov8_seg.ipynb`)
3. Notebook settings (top-right gear):
   - **Accelerator = GPU T4**
   - **Internet = ON** (needed for `pip install` + downloading `yolov8n-seg.pt`)
4. Right panel → **Input → Add Input** → attach `foodseg103-yolo2` (link above)
5. Run the code cells from top to bottom (default is **30 epochs**)
6. When it finishes: **Save Version**
7. Open the saved version → **Output** → download `best_ingredient_yolov8_seg.pt`

## Outputs
- Training run folder: `/kaggle/working/runs/segment/ingredient_yolov8_seg/`
- Copied weight for download: `/kaggle/working/best_ingredient_yolov8_seg.pt`

In [ ]:
# Step 1 — Install dependencies (run once)
import sys, subprocess

def pip_install(pkg: str):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('ultralytics>=8.2.0')
pip_install('pyyaml')

import ultralytics
print('ultralytics:', ultralytics.__version__)

In [ ]:
# Step 2 — Configure training
import os
from pathlib import Path

# Training knobs (recommended demo config for Kaggle T4)
EPOCHS = 30
IMGSZ = 640
BATCH = -1      # -1 lets Ultralytics auto-pick a safe batch size
MODEL_BASE = 'yolov8n-seg.pt'

# Optional: force a specific Kaggle input folder name if you have multiple datasets attached
# Example after attaching the dataset below: KAGGLE_DATASET_DIRNAME = 'foodseg103-yolo2'
# If you attached only ONE dataset, keep this as None (auto-detect).
KAGGLE_DATASET_DIRNAME = None

KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./')
print('WORK_DIR:', WORK_DIR)

if KAGGLE_INPUT.exists():
    print('Datasets under /kaggle/input:')
    for p in sorted(KAGGLE_INPUT.iterdir()):
        if p.is_dir():
            print(' -', p.name)
else:
    print('Not running on Kaggle (no /kaggle/input).')

In [ ]:
# Step 3 — Find a YOLOv8-seg dataset root + data.yaml
##
# Expected (preferred) YOLO layout:
#   <root>/images/train, <root>/images/val
#   <root>/labels/train, <root>/labels/val
# with labels in YOLO-seg polygon format:
#   <class_id> x1 y1 x2 y2 ... (normalized coords)
##
# If the dataset already includes a data.yaml, we will use it (best, keeps class names).
# Otherwise we generate a minimal YAML and (if we can) infer class names.

import re
import json
import yaml

# FoodSeg103 label names (no background).
# Source: HuggingFace dataset card: https://huggingface.co/datasets/EduardoPacheco/FoodSeg103
FOODSEG103_NAMES_NO_BG = [
    'candy',
    'egg tart',
    'french fries',
    'chocolate',
    'biscuit',
    'popcorn',
    'pudding',
    'ice cream',
    'cheese butter',
    'cake',
    'wine',
    'milkshake',
    'coffee',
    'juice',
    'milk',
    'tea',
    'almond',
    'red beans',
    'cashew',
    'dried cranberries',
    'soy',
    'walnut',
    'peanut',
    'egg',
    'apple',
    'date',
    'apricot',
    'avocado',
    'banana',
    'strawberry',
    'cherry',
    'blueberry',
    'raspberry',
    'mango',
    'olives',
    'peach',
    'lemon',
    'pear',
    'fig',
    'pineapple',
    'grape',
    'kiwi',
    'melon',
    'orange',
    'watermelon',
    'steak',
    'pork',
    'chicken duck',
    'sausage',
    'fried meat',
    'lamb',
    'sauce',
    'crab',
    'fish',
    'shellfish',
    'shrimp',
    'soup',
    'bread',
    'corn',
    'hamburg',
    'pizza',
    'hanamaki baozi',
    'wonton dumplings',
    'pasta',
    'noodles',
    'rice',
    'pie',
    'tofu',
    'eggplant',
    'potato',
    'garlic',
    'cauliflower',
    'tomato',
    'kelp',
    'seaweed',
    'spring onion',
    'rape',
    'ginger',
    'okra',
    'lettuce',
    'pumpkin',
    'cucumber',
    'white radish',
    'carrot',
    'asparagus',
    'bamboo shoots',
    'broccoli',
    'celery stick',
    'cilantro mint',
    'snow peas',
    'cabbage',
    'bean sprouts',
    'onion',
    'pepper',
    'green beans',
    'French beans',
    'king oyster mushroom',
    'shiitake',
    'enoki mushroom',
    'oyster mushroom',
    'white button mushroom',
    'salad',
    'other ingredients',
]

def _looks_like_yolo_root(root: Path) -> bool:
    return (root/'images'/'train').is_dir() and (root/'labels'/'train').is_dir()

def _iter_subdirs_limited(root: Path, max_depth: int = 3):
    root = root.resolve()
    for cur, dirnames, _ in os.walk(root):
        cur_p = Path(cur)
        try:
            depth = len(cur_p.relative_to(root).parts)
        except Exception:
            continue
        if depth > max_depth:
            dirnames[:] = []
            continue
        yield cur_p

def find_dataset_root() -> Path:
    if not KAGGLE_INPUT.exists():
        raise RuntimeError('This finder is meant for Kaggle. Set paths manually for local runs.')

    candidates = []
    if KAGGLE_DATASET_DIRNAME:
        p = KAGGLE_INPUT / KAGGLE_DATASET_DIRNAME
        if not p.exists():
            raise FileNotFoundError(f'KAGGLE_DATASET_DIRNAME not found: {p}')
        candidates = [p]
    else:
        candidates = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]

    for base in candidates:
        if _looks_like_yolo_root(base):
            return base
        for sub in _iter_subdirs_limited(base, max_depth=4):
            if _looks_like_yolo_root(sub):
                return sub
    raise RuntimeError('Could not find YOLO dataset root with images/train and labels/train under /kaggle/input')

def scan_nc_from_labels(labels_dir: Path, max_files: int = 8000) -> int:
    max_cls = -1
    n = 0
    for p in labels_dir.rglob('*.txt'):
        n += 1
        if n > max_files:
            break
        try:
            for line in p.read_text().strip().splitlines():
                line = line.strip()
                if not line:
                    continue
                m = re.match(r'^(\d+)(\s+|$)', line)
                if m:
                    max_cls = max(max_cls, int(m.group(1)))
        except Exception:
            continue
    if max_cls < 0:
        raise RuntimeError(f'No class ids found under {labels_dir}')
    return max_cls + 1

def try_find_names_file(root: Path):
    for cand in ['classes.txt', 'names.txt', 'class_names.txt']:
        p = root / cand
        if p.is_file():
            names = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
            if names:
                return names
    return None

def find_existing_data_yaml(root: Path):
    # Search a few levels deep for a YAML that has 'names' + 'train' keys.
    for sub in _iter_subdirs_limited(root, max_depth=4):
        for cand in ['data.yaml', 'dataset.yaml']:
            p = sub / cand
            if not p.is_file():
                continue
            try:
                obj = yaml.safe_load(p.read_text())
                if isinstance(obj, dict) and 'names' in obj and ('train' in obj or 'path' in obj):
                    return p, obj
            except Exception:
                continue
    return None, None

def _names_are_numeric_list(names_list) -> bool:
    if not isinstance(names_list, list) or not names_list:
        return False
    return all(str(x).strip().isdigit() for x in names_list)

def _names_are_numeric_dict(names_dict) -> bool:
    if not isinstance(names_dict, dict) or not names_dict:
        return False
    # values numeric (keys may be int or str)
    return all(str(v).strip().isdigit() for v in names_dict.values())

def _maybe_upgrade_foodseg103_names(names):
    # If the dataset gives numeric names but has the FoodSeg103 class count, upgrade to real names.
    if isinstance(names, list) and len(names) == len(FOODSEG103_NAMES_NO_BG) and _names_are_numeric_list(names):
        return FOODSEG103_NAMES_NO_BG
    if isinstance(names, dict) and len(names) == len(FOODSEG103_NAMES_NO_BG) and _names_are_numeric_dict(names):
        # Normalize to int-keyed dict (0..nc-1)
        out = {}
        for i, lbl in enumerate(FOODSEG103_NAMES_NO_BG):
            out[i] = lbl
        return out
    return names

DATASET_ROOT = find_dataset_root()
print('DATASET_ROOT:', DATASET_ROOT)

# Prefer an existing dataset YAML if present (keeps class names)
existing_yaml, existing_obj = find_existing_data_yaml(DATASET_ROOT)
if existing_yaml:
    DATA_YAML = existing_yaml
    names = existing_obj.get('names') if isinstance(existing_obj, dict) else None
    names_upgraded = _maybe_upgrade_foodseg103_names(names)
    if names_upgraded is not names:
        # Write a patched YAML into /kaggle/working so training uses real names
        patched_obj = dict(existing_obj)
        patched_obj['names'] = names_upgraded
        DATA_YAML = WORK_DIR / 'ingredient_seg_data.yaml'
        DATA_YAML.write_text(yaml.safe_dump(patched_obj, sort_keys=False))
        names = names_upgraded
        print('Using patched YAML with FoodSeg103 names:', DATA_YAML)
    else:
        print('Using existing YAML:', DATA_YAML)
else:
    # Create a minimal data.yaml into /kaggle/working
    nc = scan_nc_from_labels(DATASET_ROOT/'labels')
    names = try_find_names_file(DATASET_ROOT)
    if names is None or len(names) != nc:
        # If this is FoodSeg103-yolo2, upgrade numeric ids to real class names.
        if nc == len(FOODSEG103_NAMES_NO_BG):
            names = FOODSEG103_NAMES_NO_BG
            print('✓ Using FoodSeg103 class names (no background)')
        else:
            names = [str(i) for i in range(nc)]
            print('⚠ Could not infer class names; using numeric names 0..nc-1')

    data_obj = {
        'path': str(DATASET_ROOT),
        'train': 'images/train',
        'val': 'images/val' if (DATASET_ROOT/'images'/'val').is_dir() else 'images/train',
        'names': names,
    }
    DATA_YAML = WORK_DIR / 'ingredient_seg_data.yaml'
    DATA_YAML.write_text(yaml.safe_dump(data_obj, sort_keys=False))
    print('Wrote YAML:', DATA_YAML)

# Quick sanity checks
assert (DATASET_ROOT/'images'/'train').is_dir(), 'missing images/train'
assert (DATASET_ROOT/'labels'/'train').is_dir(), 'missing labels/train'
print('✓ Dataset looks like YOLOv8-seg format')

# Export a label mapping JSON for the app (download from Kaggle Output)
labels_json = WORK_DIR / 'ingredient_labels.json'
if isinstance(names, dict):
    # Some yamls store names as {id: name}
    labels_json.write_text(json.dumps({str(k): v for k, v in names.items()}, indent=2))
elif isinstance(names, list):
    labels_json.write_text(json.dumps({str(i): n for i, n in enumerate(names)}, indent=2))
else:
    labels_json.write_text(json.dumps({}, indent=2))
print('✓ Saved label mapping →', labels_json)

In [ ]:
# Step 4 — Train YOLOv8-seg (instance segmentation)
# This writes outputs into /kaggle/working so Kaggle can export them.

from ultralytics import YOLO

print('Loading base model:', MODEL_BASE)
model = YOLO(MODEL_BASE)

results = model.train(
    data=str(DATA_YAML),
    task='segment',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(WORK_DIR / 'runs'),
    name='ingredient_yolov8_seg',
    exist_ok=True,
    pretrained=True
)

print('✓ Training complete')

In [ ]:
# Step 5 — Export/copy the best weights into a simple filename for download
from pathlib import Path
import shutil

run_dir = WORK_DIR / 'runs' / 'segment' / 'ingredient_yolov8_seg'
best = run_dir / 'weights' / 'best.pt'
last = run_dir / 'weights' / 'last.pt'
print('Run dir:', run_dir)
print('Best exists:', best.exists(), best)
print('Last exists:', last.exists(), last)

out_best = WORK_DIR / 'best_ingredient_yolov8_seg.pt'
if best.exists():
    shutil.copy2(best, out_best)
    print('✓ Copied →', out_best)
else:
    print('⚠ best.pt not found — check the training output above')

# List output files (what Kaggle will show under Output)
for p in sorted(WORK_DIR.glob('*.pt')):
    print('Output weight:', p.name, f'({p.stat().st_size/1e6:.1f} MB)')

## After training: download + use locally

### Download from Kaggle Output
1. Click **Save Version**
2. Open the saved version
3. Right panel → **Output** → download:
- `best_ingredient_yolov8_seg.pt`
- `ingredient_labels.json`  (this is important — it fixes numeric class names like 57 → real labels)

### Put it into this repo
Copy both files into the repo `models/` folder:
- `models/best_ingredient_yolov8_seg.pt`
- `models/ingredient_labels.json`

### Note
The app will use `ingredient_labels.json` to display readable ingredient/component names and to do USDA lookups per ingredient.